<a href="https://colab.research.google.com/github/karkhutmaria/KarkhutM_topical_compling_questions/blob/main/KarkhutM_03_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install nltk sentence-transformers scikit-learn pandas matplotlib networkx
import nltk
nltk.download('wordnet')
from nltk.corpus import wordnet as wn
from nltk.tokenize import word_tokenize
nltk.download('all')
from nltk.corpus import stopwords
nltk.download('stopwords')
stopwords = set(stopwords.words('english'))
import re
word_re = re.compile(r"^[a-z]+(?:'[a-z]+)?$")
from collections import defaultdict
from sentence_transformers import SentenceTransformer, util

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger

# Задание 1 (5 балла)

Имплементируйте алгоритм Леска (описание есть в семинаре) и оцените качество его работы на датасете `data/corpus_wsd_50k.txt`

В качестве метрики близости вы должны попробовать два подхода:

1) Jaccard score на множествах слов (определений и контекста)
2) Cosine distance на эмбедингах sentence_transformers

В качестве метрики используйте accuracy (% правильных ответов). Предсказывайте только многозначные слова в датасете

Контекст вы можете определить самостоятельно (окно вокруг целевого слова или все предложение). Также можете поэкспериментировать с предобработкой для обоих методов.

In [2]:
corpus_wsd = []
with open("corpus_wsd_50k.txt", encoding="utf-8") as file:
  raw = file.read()

for sent in raw.split("\n\n"):
  sent = sent.strip("\n")
  if not sent:
    continue

  tokens = []
  for line in sent.split("\n"):
    parts = line.split("\t")

    if len(parts) == 3:
      sense, lemma, surface = parts
    elif len(parts) == 2:
      sense, lemma, surface = "", parts[0], parts[1]
    else:
      continue

    tokens.append({
      "sense":   sense.strip(),
      "lemma":   lemma.strip().lower(),
      "surface": surface.strip().lower(),
      })

  if tokens:
    corpus_wsd.append(tokens)

corpus_wsd[0]

[{'sense': '', 'lemma': 'how', 'surface': 'how'},
 {'sense': 'long%3:00:02::', 'lemma': 'long', 'surface': 'long'},
 {'sense': '', 'lemma': 'have', 'surface': 'has'},
 {'sense': '', 'lemma': 'it', 'surface': 'it'},
 {'sense': 'be%2:42:03::', 'lemma': 'be', 'surface': 'been'},
 {'sense': '', 'lemma': 'since', 'surface': 'since'},
 {'sense': '', 'lemma': 'you', 'surface': 'you'},
 {'sense': 'review%2:31:00::', 'lemma': 'review', 'surface': 'reviewed'},
 {'sense': '', 'lemma': 'the', 'surface': 'the'},
 {'sense': 'objective%1:09:00::',
  'lemma': 'objective',
  'surface': 'objectives'},
 {'sense': '', 'lemma': 'of', 'surface': 'of'},
 {'sense': '', 'lemma': 'you', 'surface': 'your'},
 {'sense': 'benefit%1:21:00::', 'lemma': 'benefit', 'surface': 'benefit'},
 {'sense': '', 'lemma': 'and', 'surface': 'and'},
 {'sense': 'service%1:04:07::', 'lemma': 'service', 'surface': 'service'},
 {'sense': 'program%1:09:01::', 'lemma': 'program', 'surface': 'program'},
 {'sense': '', 'lemma': '?', 'surfa

In [3]:
def get_context(sentence, keyword, window=None, use_surface=False):
  col = "surface" if use_surface else "lemma"
  toks = [t[col] for t in sentence]

  if window is None:
    return [w for w in toks if w != keyword]

  positions = [i for i, w in enumerate(toks) if w == keyword]
  context = []
  for i in positions:
    for j in range(max(0, i - window), min(len(toks), i + window + 1)):
      if j != i:
        context.append(toks[j])
  return context

In [4]:
def clean_definitions(definition):
  return {t for t in word_tokenize(definition.lower())
          if t not in stopwords and word_re.match(t)}


def lesk_jaccard(sentence, keyword, window=None):
  synsets = wn.synsets(keyword)
  if not synsets:
    return None

  ctx = {t for t in get_context(sentence, keyword, window)
        if t not in stopwords and word_re.match(t)}

  best_syn, best_score = None, -1.0
  for syn in synsets:
    def_set = clean_definitions(syn.definition())
    if not ctx or not def_set:
      score = 0.0
    else:
      score = len(ctx & def_set) / len(ctx | def_set)
    if score > best_score:
      best_score, best_syn = score, syn
  return best_syn

In [13]:
model = SentenceTransformer("all-MiniLM-L6-v2")
_def_cache = {}

def lesk_cosine(sentence, keyword, window=None):
    synsets = wn.synsets(keyword)
    if not synsets:
        return None

    ctx = get_context(sentence, keyword, window, use_surface=True)
    ctx_str = " ".join(ctx) if ctx else keyword
    ctx_emb = model.encode([ctx_str], convert_to_tensor=True,
                           normalize_embeddings=True)

    key = tuple(s.name() for s in synsets)      # <-- tuple, НЕ list
    if key not in _def_cache:
        _def_cache[key] = model.encode(
            [s.definition() for s in synsets],
            convert_to_tensor=True,
            normalize_embeddings=True,
        )
    def_embs = _def_cache[key]

    scores = util.cos_sim(ctx_emb, def_embs)[0]
    return synsets[int(scores.argmax())]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [22]:
def syn_to_key(syn, lemma):
  for lem in syn.lemmas():
    if lem.name().lower() == lemma:
      return lem.key()
  return syn.lemmas()[0].key()

def evaluate(lesk_fn, window=None, name="Lesk"):

  correct = total = 0

  corpus_wsd_cut = corpus_wsd[:1000]

  for sent in corpus_wsd_cut:
    for tok in sent:
      lemma, gold = tok["lemma"], tok["sense"]
      if not gold or len(wn.synsets(lemma)) < 2:
        continue
      pred = lesk_fn(sent, lemma, window=window)
      total += 1
      if pred is not None and syn_to_key(pred, lemma) == gold:
        correct += 1

  acc = correct / total if total else 0.0
  print(f"[{name}, window={window}] accuracy = {correct}/{total} = {acc:.4f}")

In [23]:
for w in [5, 10, None]:
    evaluate(lesk_jaccard, window=w, name="Jaccard")

for w in [5, 10, None]:
    evaluate(lesk_cosine, window=w, name="Cosine")

[Jaccard, window=5] accuracy = 3392/7573 = 0.4479
[Jaccard, window=10] accuracy = 3324/7573 = 0.4389
[Jaccard, window=None] accuracy = 3274/7573 = 0.4323
[Cosine, window=5] accuracy = 2267/7573 = 0.2994
[Cosine, window=10] accuracy = 2299/7573 = 0.3036
[Cosine, window=None] accuracy = 2302/7573 = 0.3040


# Задание 2 (5 балла)
Попробуйте разные алгоритмы кластеризации на датасете - `https://github.com/nlpub/russe-wsi-kit/blob/initial/data/main/wiki-wiki/train.csv`

Используйте код из семинара как основу. Используйте ARI как метрику качества.

Попробуйте все 4 алгоритма кластеризации, про которые говорилось на семинаре. Для каждого из алгоритмов попробуйте настраивать гиперпараметры (посмотрите их в документации). Прогоните как минимум 5 экспериментов (не обязательно успешных) с разными параметрами на каждый алгоритме кластеризации и оцените: качество кластеризации, скорость работы, интуитивность параметров.

Помимо этого также выберите 1 дополнительный алгоритм кластеризации отсюда - https://scikit-learn.org/stable/modules/clustering.html , опишите своими словами принцип его работы  и проделайте аналогичные эксперименты.

In [27]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/nlpub/russe-wsi-kit/initial/data/main/wiki-wiki/train.csv', sep='\t')
df.head(100)

,context_id,word,gold_sense_id,predict_sense_id,positions,context
0,1,замок,1,NaN,"0-5, 339-344",замок владимира мономаха в любече . многочисле...
1,2,замок,1,NaN,"11-16, 17-22, 188-193","шильонский замок замок шильйон ( ) , известный..."
2,3,замок,1,NaN,299-304,проведения архитектурно - археологических рабо...
3,4,замок,1,NaN,111-116,"топи с . , л . белокуров легенда о завещании м..."
4,5,замок,1,NaN,"134-139, 262-267",великий князь литовский гедимин после успешной...
...,...,...,...,...,...,...
95,96,замок,1,NaN,"163-168, 213-218, 364-369",без каминов и очагов . у входа в башню было по...
96,97,замок,1,NaN,"221-226, 276-281",v . в том сражении шотландцы потерпели сокруши...
97,98,замок,1,NaN,"0-5, 16-21, 179-184",замок данноттар замок данноттар ( ) расположен...
98,99,замок,1,NaN,232-237,. благодаря поэме байрона заключение бонивара ...


In [28]:
grouped_df = df.groupby('word')[['word', 'context', 'gold_sense_id']]
for key, _ in grouped_df:
    print(grouped_df.get_group(key), "\n\n")
    break

    word                                            context  gold_sense_id
383  бор  бор ( элемент ) бор — элемент тринадцатой груп...              1
384  бор  бор - углерод - кремний семейство сплавов на о...              1
385  бор  с большим выделением теплоты , образуется окси...              1
386  бор  это объясняется прежде всего тем , что у компл...              1
387  бор  совсем малочисленна . элементарный бор в приро...              1
388  бор  действующем при месторождении горно - химическ...              1
389  бор  b c ) . при нагревании в атмосфере кислорода и...              1
390  бор  собственных минералов бора в чужих минералах о...              1
391  бор  бор - углерод - кремний семейство сплавов на о...              1
392  бор  с другими галогенами с образованием тригалоген...              1
393  бор  и в стабильны и входят в состав природного бор...              1
394  бор  номером . обозначается символом b ( ) . в своб...              1
395  бор  и взаимные пере

In [40]:
from sklearn.cluster import KMeans, DBSCAN, AffinityPropagation, AgglomerativeClustering
import numpy as np
from sklearn.metrics import adjusted_rand_score

from IPython.display import Image
from IPython.core.display import HTML

DBSCAN


In [37]:
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

ARI = []

for key, _ in grouped_df:
    texts = grouped_df.get_group(key)['context'].values

    X = model.encode(
        list(texts),
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    cluster = DBSCAN(eps=0.5, min_samples=2)
    labels = cluster.fit_predict(X) + 1

    gold = grouped_df.get_group(key)['gold_sense_id']
    ARI.append(adjusted_rand_score(gold, labels))

print(np.mean(ARI))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

0.037370937178974294


Affinity Propagation

In [38]:
ARI = []

for key, _ in grouped_df:
  texts = grouped_df.get_group(key)['context'].values

  X = model.encode(
        list(texts),
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


  cluster = AffinityPropagation(damping=0.6, max_iter=100)
  labels = cluster.fit_predict(X) + 1

  ARI.append(adjusted_rand_score(grouped_df.get_group(key)['gold_sense_id'], labels))

print(np.mean(ARI))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

0.042740969848549505


Agglomerative clustering

In [42]:
ARI = []

for key, _ in grouped_df:
  texts = grouped_df.get_group(key)['context'].values

  X = model.encode(
        list(texts),
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


  cluster = AgglomerativeClustering(n_clusters=3, linkage="average")
  labels = cluster.fit_predict(X) + 1

  ARI.append(adjusted_rand_score(grouped_df.get_group(key)['gold_sense_id'], labels))

print(np.mean(ARI))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

-0.009158236277409101


Kmeans

In [43]:
ARI = []

for key, _ in grouped_df:
  texts = grouped_df.get_group(key)['context'].values

  X = model.encode(
        list(texts),
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


  cluster = KMeans(n_clusters=5, max_iter=200)
  labels = cluster.fit_predict(X) + 1

  ARI.append(adjusted_rand_score(grouped_df.get_group(key)['gold_sense_id'], labels))

print(np.mean(ARI))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

0.09568829406859249


Birch

In [44]:
from sklearn.cluster import Birch

ARI = []

for key, _ in grouped_df:
  texts = grouped_df.get_group(key)['context'].values

  X = model.encode(
        list(texts),
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )


  cluster = Birch(n_clusters=6)
  labels = cluster.fit_predict(X) + 1

  ARI.append(adjusted_rand_score(grouped_df.get_group(key)['gold_sense_id'], labels))

print(np.mean(ARI))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/sklearn/cluster/_birch.py:727: ConvergenceWarning: Number of subclusters found (5) by BIRCH is less than (6). Decrease the threshold.
  warnings.warn(


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/sklearn/cluster/_birch.py:727: ConvergenceWarning: Number of subclusters found (2) by BIRCH is less than (6). Decrease the threshold.
  warnings.warn(


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

0.04773769732740768


В целом, эксперименты с разными инструментами кластеризации дали очень разные результаты. Какие-то более успешные, какие-то менее. Скорость кластеризации без разделения на батчи оставляла желать лучшего, поэтому я во всех экспериментах применила батчи. Не могу сказать, что параметры кластеризаторов прям очень интуитивные, но параметры n_clusters и max_iter вполне понятны.